In [ ]:
# CS506 Final Project - Playlist-based Music Recommendation
# =========================================================
# This notebook covers: data loading, cleaning, feature extraction,
# visualization, and preliminary modeling for the March check-in.

import os
import csv
import re
import math
import time
import random
import unicodedata
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Configuration
DATA_PATH = "spotify_dataset.csv"
MIN_PLAYLIST_SIZE = 5
MAX_PLAYLIST_SIZE = 500
MIN_SONG_FREQ = 3
TOPK = 10
TEST_FRAC = 0.2

## 1. Data Loading

We load the Spotify Playlists dataset from Kaggle. The raw CSV has some malformed rows which we handle with a robust loader.

In [ ]:
def clean_col(c: str) -> str:
    c = str(c).strip()
    if c.startswith('"') and c.endswith('"') and len(c) >= 2:
        c = c[1:-1].strip()
    return c

def robust_load_csv(path: str) -> pd.DataFrame:
    """Load CSV, dropping rows with wrong number of columns."""
    with open(path, "r", encoding="utf-8", errors="replace", newline="") as f:
        header_line = f.readline().rstrip("\n")
    raw_header = next(csv.reader([header_line], delimiter=",", quotechar='"', escapechar="\\"))
    expected_cols = len(raw_header)
    header = [clean_col(c) for c in raw_header]

    good_rows, bad_count = [], 0
    with open(path, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f, delimiter=",", quotechar='"', escapechar="\\")
        next(reader)  # skip header
        for row in reader:
            if len(row) == expected_cols:
                good_rows.append(row)
            else:
                bad_count += 1

    df = pd.DataFrame(good_rows, columns=header)
    print(f"Loaded shape: {df.shape}")
    print(f"Malformed rows dropped: {bad_count}")
    print(f"Columns: {list(df.columns)}")
    return df

# Download data if needed
if not os.path.exists(DATA_PATH):
    print("Data file not found. Run: python download_data.py")
else:
    raw_df = robust_load_csv(DATA_PATH)
    raw_df.head()

## 2. Data Cleaning

**Cleaning steps:**
1. Normalize artist and track names (lowercase, remove accents, collapse whitespace)
2. Create unique keys for songs (`artist — track`) and playlists (`user :: playlist_name`)
3. Remove rows with empty/broken fields
4. Remove duplicate (playlist, song) pairs

In [ ]:
USER_COL = "user_id"
ARTIST_COL = "artistname"
TRACK_COL = "trackname"
PLAYLIST_COL = "playlistname"

def normalize_text(s: str) -> str:
    """Normalize text: lowercase, remove accents/special chars, collapse spaces."""
    if pd.isna(s):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower().strip()
    s = s.replace("&", " and ")
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

work_df = raw_df.copy()
for col in [USER_COL, ARTIST_COL, TRACK_COL, PLAYLIST_COL]:
    work_df[col] = work_df[col].fillna("").astype(str)

work_df["artist_norm"] = work_df[ARTIST_COL].map(normalize_text)
work_df["track_norm"] = work_df[TRACK_COL].map(normalize_text)
work_df["song_key"] = work_df["artist_norm"] + " — " + work_df["track_norm"]
work_df["playlist_key"] = work_df[USER_COL].astype(str) + " :: " + work_df[PLAYLIST_COL].map(normalize_text)

# Remove empty/broken records
before = len(work_df)
work_df = work_df[
    (work_df["song_key"].str.strip() != "—") &
    (work_df["artist_norm"].str.strip() != "") &
    (work_df["track_norm"].str.strip() != "") &
    (work_df["playlist_key"].str.strip() != "")
].copy()
print(f"Removed empty/broken rows: {before - len(work_df):,}")

# Remove duplicate (playlist, song) pairs
before = len(work_df)
work_df = work_df.drop_duplicates(subset=["playlist_key", "song_key"], keep="first").copy()
print(f"Removed duplicate (playlist, song) rows: {before - len(work_df):,}")
print(f"\nCleaned dataset shape: {work_df.shape}")
print(f"Unique playlists: {work_df['playlist_key'].nunique():,}")
print(f"Unique songs: {work_df['song_key'].nunique():,}")
print(f"Unique users: {work_df[USER_COL].nunique():,}")

## 3. Feature Extraction

We compute several features for filtering and analysis:
- **Song popularity** (`song_freq`): how many playlists a song appears in
- **Playlist size** (`playlist_size`): number of unique songs per playlist
- **Artist count** (`playlist_artist_count`): number of distinct artists per playlist
- **Artist diversity ratio**: artist count / playlist size (higher = more diverse)

In [ ]:
# Song popularity (number of playlists containing this song)
song_freq = work_df["song_key"].value_counts()
work_df["song_freq"] = work_df["song_key"].map(song_freq)

# Playlist size (unique songs per playlist)
playlist_size = work_df.groupby("playlist_key")["song_key"].nunique()
work_df["playlist_size"] = work_df["playlist_key"].map(playlist_size)

# Artist count per playlist
playlist_artist_count = work_df.groupby("playlist_key")["artist_norm"].nunique()
work_df["playlist_artist_count"] = work_df["playlist_key"].map(playlist_artist_count)

# Artist diversity ratio
work_df["artist_diversity_ratio"] = work_df["playlist_artist_count"] / work_df["playlist_size"].clip(lower=1)

# Filter playlists and songs
filtered = work_df[
    (work_df["playlist_size"] >= MIN_PLAYLIST_SIZE) &
    (work_df["playlist_size"] <= MAX_PLAYLIST_SIZE) &
    (work_df["song_freq"] >= MIN_SONG_FREQ)
].copy()

# Recompute sizes after song filtering and keep only playlists still large enough
filtered_playlist_size = filtered.groupby("playlist_key")["song_key"].nunique()
filtered = filtered[filtered["playlist_key"].isin(filtered_playlist_size.index)].copy()
filtered["playlist_size_filtered"] = filtered["playlist_key"].map(filtered_playlist_size)
filtered = filtered[filtered["playlist_size_filtered"] >= MIN_PLAYLIST_SIZE].copy()

print(f"Shape after filtering: {filtered.shape}")
print(f"Unique playlists: {filtered['playlist_key'].nunique():,}")
print(f"Unique songs: {filtered['song_key'].nunique():,}")
print(f"Unique users: {filtered[USER_COL].nunique():,}")

# Build playlist-level feature table
playlist_features = (
    filtered.groupby("playlist_key")
    .agg(
        user_id=(USER_COL, "first"),
        playlist_name=(PLAYLIST_COL, "first"),
        n_songs=("song_key", "nunique"),
        n_artists=("artist_norm", "nunique"),
        avg_song_popularity=("song_freq", "mean"),
    )
    .reset_index()
)
playlist_features["artist_diversity_ratio"] = playlist_features["n_artists"] / playlist_features["n_songs"].clip(lower=1)
print("\nPlaylist feature summary:")
playlist_features.describe().round(2)

## 4. Data Visualizations

We present several visualizations to understand the data distribution and inform modeling decisions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribution of playlist sizes
axes[0, 0].hist(playlist_features["n_songs"], bins=40, edgecolor="black", alpha=0.7, color="steelblue")
axes[0, 0].set_xlabel("Number of Songs")
axes[0, 0].set_ylabel("Number of Playlists")
axes[0, 0].set_title("Distribution of Playlist Sizes")
axes[0, 0].axvline(playlist_features["n_songs"].median(), color="red", linestyle="--", label=f'Median={playlist_features["n_songs"].median():.0f}')
axes[0, 0].legend()

# 2. Song popularity distribution (log scale)
song_pop = filtered["song_key"].value_counts()
axes[0, 1].hist(song_pop.values, bins=50, edgecolor="black", alpha=0.7, color="coral")
axes[0, 1].set_xlabel("Number of Playlists Containing Song")
axes[0, 1].set_ylabel("Number of Songs")
axes[0, 1].set_title("Song Popularity Distribution")
axes[0, 1].set_yscale("log")

# 3. Top 15 most frequent artists
top_artists = filtered[ARTIST_COL].value_counts().head(15).sort_values()
axes[1, 0].barh(top_artists.index, top_artists.values, color="mediumseagreen", edgecolor="black")
axes[1, 0].set_xlabel("Occurrences Across Playlists")
axes[1, 0].set_title("Top 15 Most Frequent Artists")

# 4. Artist diversity ratio distribution
axes[1, 1].hist(playlist_features["artist_diversity_ratio"], bins=30, edgecolor="black", alpha=0.7, color="orchid")
axes[1, 1].set_xlabel("Artist Diversity Ratio (unique artists / playlist size)")
axes[1, 1].set_ylabel("Number of Playlists")
axes[1, 1].set_title("Playlist Artist Diversity")

plt.tight_layout()
plt.savefig("visualizations.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved visualizations.png")

In [ ]:
# Additional visualization: Top 15 most common playlist names
top_playlist_names = playlist_features["playlist_name"].value_counts().head(15).sort_values()
plt.figure(figsize=(10, 6))
top_playlist_names.plot(kind="barh", color="steelblue", edgecolor="black")
plt.xlabel("Number of Playlists")
plt.ylabel("Playlist Name")
plt.title("Top 15 Most Common Playlist Names")
plt.tight_layout()
plt.show()

# Scatter: playlist size vs artist diversity
plt.figure(figsize=(8, 5))
plt.scatter(playlist_features["n_songs"], playlist_features["artist_diversity_ratio"],
            alpha=0.4, s=20, color="teal")
plt.xlabel("Playlist Size (# songs)")
plt.ylabel("Artist Diversity Ratio")
plt.title("Playlist Size vs. Artist Diversity")
plt.tight_layout()
plt.show()

## 5. Train/Test Split & Masked Test Cases

We split playlists into train (80%) and test (20%). For evaluation, we mask (hide) a portion of songs from each test playlist and try to predict them.

In [ ]:
# Build playlist -> songs mapping
playlist_to_songs = filtered.groupby("playlist_key")["song_key"].apply(list).to_dict()

# Deduplicate within each playlist (preserve order)
for pk in list(playlist_to_songs.keys()):
    seen = set()
    deduped = []
    for s in playlist_to_songs[pk]:
        if s not in seen:
            deduped.append(s)
            seen.add(s)
    playlist_to_songs[pk] = deduped

# Song display name mapping
song_display_map = (
    filtered.groupby("song_key")[[ARTIST_COL, TRACK_COL]]
    .first()
    .assign(display_name=lambda x: x[ARTIST_COL].astype(str) + " — " + x[TRACK_COL].astype(str))
)["display_name"].to_dict()

# Train/test split by playlist
all_playlist_keys = list(playlist_to_songs.keys())
random.shuffle(all_playlist_keys)
n_test = max(1, int(len(all_playlist_keys) * TEST_FRAC))

test_playlists = set(all_playlist_keys[:n_test])
train_playlists = set(all_playlist_keys[n_test:])
train_playlist_to_songs = {k: playlist_to_songs[k] for k in train_playlists}
test_playlist_to_songs = {k: playlist_to_songs[k] for k in test_playlists}

print(f"Train playlists: {len(train_playlist_to_songs):,}")
print(f"Test playlists:  {len(test_playlist_to_songs):,}")

# Create masked test cases
def make_test_cases(test_dict, min_input_size=2, max_hide=3):
    cases = []
    for pk, songs in test_dict.items():
        if len(songs) < (min_input_size + 1):
            continue
        n_hide = max(1, min(max_hide, int(round(0.2 * len(songs)))))
        if len(songs) - n_hide < min_input_size:
            n_hide = len(songs) - min_input_size
        if n_hide <= 0:
            continue
        hidden = set(random.sample(songs, n_hide))
        observed = [s for s in songs if s not in hidden]
        if len(observed) >= min_input_size and len(hidden) > 0:
            cases.append({"playlist_key": pk, "observed": observed, "hidden": hidden})
    return cases

test_cases = make_test_cases(test_playlist_to_songs)
print(f"Usable masked test cases: {len(test_cases):,}")

## 6. Preliminary Models

We implement two baseline models:

1. **Popularity Baseline**: Recommends the most globally popular songs (ignoring playlist context). This is the simplest possible recommender.
2. **Item-Item Co-occurrence**: For each observed song, finds playlists containing it, then ranks candidate songs by how often they co-occur with the observed set. This captures playlist-level context.

**Why these models?** The popularity baseline sets a lower bound — any useful model must beat simply recommending popular songs. Co-occurrence is a lightweight collaborative filtering approach that exploits the structure of playlists without requiring matrix factorization.

In [ ]:
# --- Popularity Baseline ---
train_song_popularity = Counter()
for songs in train_playlist_to_songs.values():
    train_song_popularity.update(songs)
popular_ranking = [song for song, _ in train_song_popularity.most_common()]

def recommend_popularity(observed_songs, topk=10):
    seen = set(observed_songs)
    return [s for s in popular_ranking if s not in seen][:topk]

# --- Item-Item Co-occurrence ---
song_to_playlists = defaultdict(set)
for pk, songs in train_playlist_to_songs.items():
    for s in songs:
        song_to_playlists[s].add(pk)

playlist_song_set = {pk: set(songs) for pk, songs in train_playlist_to_songs.items()}

def recommend_cooccurrence(observed_songs, topk=10):
    observed = set(observed_songs)
    candidate_scores = defaultdict(float)

    for s in observed:
        for pk in song_to_playlists.get(s, set()):
            for candidate in playlist_song_set[pk]:
                if candidate not in observed:
                    candidate_scores[candidate] += 1.0

    if not candidate_scores:
        return recommend_popularity(observed_songs, topk=topk)

    ranked = sorted(candidate_scores.items(), key=lambda x: (-x[1], -train_song_popularity[x[0]], x[0]))
    recs = [song for song, _ in ranked[:topk]]

    # Backfill with popularity if fewer than topk
    if len(recs) < topk:
        seen = observed | set(recs)
        recs.extend([s for s in popular_ranking if s not in seen][:topk - len(recs)])

    return recs[:topk]

print("Models built successfully.")

## 7. Evaluation & Results

We evaluate both models using three ranking metrics:
- **Hit Rate @ K**: Fraction of test cases where at least one hidden song appears in the top-K recommendations
- **Recall @ K**: Average fraction of hidden songs recovered in the top-K
- **MRR @ K**: Mean Reciprocal Rank — how high the first correct prediction ranks

In [ ]:
def evaluate_model(test_cases, recommender_fn, topk=10):
    hits, recalls, reciprocal_ranks = [], [], []
    for case in test_cases:
        observed, hidden = case["observed"], case["hidden"]
        recs = recommender_fn(observed, topk=topk)
        rec_set = set(recs)
        n_hit = len(hidden & rec_set)

        hits.append(1.0 if n_hit > 0 else 0.0)
        recalls.append(n_hit / len(hidden))

        rr = 0.0
        for rank, item in enumerate(recs, start=1):
            if item in hidden:
                rr = 1.0 / rank
                break
        reciprocal_ranks.append(rr)

    return {
        f"HitRate@{topk}": float(np.mean(hits)),
        f"Recall@{topk}": float(np.mean(recalls)),
        f"MRR@{topk}": float(np.mean(reciprocal_ranks)),
        "num_test_cases": len(test_cases),
    }

start_eval = time.time()
pop_metrics = evaluate_model(test_cases, recommend_popularity, topk=TOPK)
co_metrics = evaluate_model(test_cases, recommend_cooccurrence, topk=TOPK)
print(f"Evaluation finished in {time.time() - start_eval:.2f}s\n")

results_df = pd.DataFrame([
    {"Model": "Popularity Baseline", **pop_metrics},
    {"Model": "Co-occurrence", **co_metrics},
])
results_df

In [ ]:
# Visualize model comparison
metric_cols = [f"HitRate@{TOPK}", f"Recall@{TOPK}", f"MRR@{TOPK}"]
plot_df = results_df.set_index("Model")[metric_cols]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(metric_cols))
width = 0.3

bars1 = ax.bar(x - width/2, plot_df.iloc[0].values, width, label="Popularity Baseline", color="steelblue", edgecolor="black")
bars2 = ax.bar(x + width/2, plot_df.iloc[1].values, width, label="Co-occurrence", color="coral", edgecolor="black")

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 4), textcoords="offset points", ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(metric_cols)
ax.set_ylabel("Score")
ax.set_title(f"Model Comparison at K={TOPK}")
ax.set_ylim(0, max(0.05, plot_df.max().max() * 1.3))
ax.legend()
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Example Recommendations

Below we show a few concrete examples of how each model recommends songs for masked test playlists.

In [ ]:
def pretty_song(song_key):
    return song_display_map.get(song_key, song_key)

for i, case in enumerate(test_cases[:5], start=1):
    observed, hidden = case["observed"], case["hidden"]
    pop_recs = recommend_popularity(observed, topk=TOPK)
    co_recs = recommend_cooccurrence(observed, topk=TOPK)

    print(f"--- Example {i} ---")
    print("Observed:", ", ".join(pretty_song(s) for s in observed[:6]),
          "..." if len(observed) > 6 else "")
    print("Hidden:  ", ", ".join(pretty_song(s) for s in hidden))
    print("Pop recs:", ", ".join(pretty_song(s) for s in pop_recs[:5]))
    print("Co-oc:   ", ", ".join(pretty_song(s) for s in co_recs[:5]))
    
    # Check if co-occurrence found any hidden song
    co_hits = hidden & set(co_recs)
    if co_hits:
        print(f"  -> Co-occurrence HIT: {', '.join(pretty_song(s) for s in co_hits)}")
    print()

## 9. Summary & Interpretation

**Data Processing:**
- Loaded 267K+ rows from the Spotify Playlists dataset
- Cleaned text (normalization, deduplication), removed broken rows
- Filtered to playlists with 5-500 songs and songs appearing in 3+ playlists

**Features Extracted:**
1. `song_freq` — track popularity (playlist count)
2. `playlist_size` — number of songs per playlist
3. `playlist_artist_count` — number of distinct artists per playlist
4. `artist_diversity_ratio` — artist diversity within a playlist

**Models Tested:**
1. Popularity Baseline — recommends globally popular songs
2. Item-Item Co-occurrence — recommends songs that frequently appear in the same playlists as the observed songs

**Interpretation:**
The co-occurrence model captures playlist context (e.g., genre coherence) that the popularity baseline misses entirely. This confirms that playlist co-occurrence contains useful collaborative signal. However, both models are still preliminary — next steps include matrix factorization and potentially content-based features from lyrics.